# 01 - DeepURLBench audit and fixed chronological split

Attach the project as a Kaggle Dataset and enable Internet. This notebook reads the source once, checks the instructor-required dates, and writes reusable Parquet files only under `/kaggle/working`. It exits without changing the date ranges if coverage is insufficient.

In [ ]:
# Set this to the directory name of the attached project Kaggle Dataset.
PROJECT_INPUT = '/kaggle/input/charcnn-project'
WORKING_PROJECT = '/kaggle/working/charcnn'
import os, zipfile
os.makedirs(WORKING_PROJECT, exist_ok=True)
zip_path = f'{PROJECT_INPUT}/charcnn-project.zip'
if not os.path.exists(zip_path):
    raise FileNotFoundError(f'Expected {zip_path}. Upload the project ZIP as instructed.')
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(WORKING_PROJECT)
%pip install -q -r {WORKING_PROJECT}/requirements.txt


In [ ]:
import os, sys
os.chdir(WORKING_PROJECT)
sys.path.insert(0, f'{WORKING_PROJECT}/src')
from charcnn.audit_split import main
main(['--config', f'{WORKING_PROJECT}/configs/config.yaml'])


In [ ]:
import json
from pathlib import Path
data_dir = Path('/kaggle/working/charcnn/data')
print((data_dir / 'audit.json').read_text())
print((data_dir / 'manifest.json').read_text())


## Save only the prepared dataset

The module removes its temporary raw staging file and Hugging Face cache automatically after a successful run. The next cell is only for an older run that created those files before this project update.

In [ ]:
# Safe cleanup for an already-completed older run. It preserves train, validation, test, and metadata.
import shutil
from pathlib import Path
root = Path('/kaggle/working/charcnn')
(root / 'data' / 'source_url_date_label.parquet').unlink(missing_ok=True)
shutil.rmtree(root / 'hf_cache', ignore_errors=True)
print('Prepared files to publish:')
for path in sorted((root / 'data').iterdir()):
    print(path.name, f'{path.stat().st_size / 1024**2:.1f} MB')
